In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/biohub_embryo.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))


In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/biohub_embryo.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))


In [ ]:
import os
BIOHUB_PRESET = 'dual_seed_near_balanced_center_confirmed_synthetic_gap'
BIOHUB_SCORE_AXIS = 'near-balanced shared detections with marginal synthetic-gap confirmation'

# Parameters for constrained lineage reconstruction.
os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.96875"
os.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = '1.0'
os.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"
os.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "1.5"
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_GAP_CLOSE_UM"] = "5.8"
os.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"
os.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"
os.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"
os.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"
os.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "0"
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "4.66"
os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = '8.5'
os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "7.65"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "0"
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "500"
os.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = '0'
os.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"

print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)

# --- factory env overrides ---
os.environ["H1R_KERNEL"] = '1'
os.environ["H1R_EDGE_EPOCHS"] = '1'
os.environ["H1R_EDGE_BS"] = '1'
os.environ["H1R_EDGE_LR"] = '1e-4'
os.environ["H1R_EDGE_MAX_STEPS"] = '2'
os.environ["H1R_EDGE_NODE_CAP"] = '256'
os.environ["H1R_EDGE_VAL_MOD"] = '6'
os.environ["H1R_EDGE_AMP"] = '1'
os.environ["H1R_EDGE_RESUME"] = '0'
os.environ["H1R_EDGE_SEED"] = '0'
os.environ["H1R_APPEARANCE_DIM"] = '64'
os.environ["H1R_TRIPLET_WEIGHT"] = '0.05'


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"
GAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))
GAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))
GAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))

# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.
USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/checkpoint_last.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,
    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,
    "gap_density_gain": GAP_DENSITY_GAIN,
    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,
    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,
    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


# Resolve the independent-seed pack by checksum, then materialize only its weights.
import hashlib as _hashlib

_secondary_manifest_explicit = Path(os.environ.get(
    "BIOHUB_SECONDARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json",
))
_secondary_expected_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
_secondary_slug = "biohub-temporal-unet3d-seed314159-v1"


def _find_secondary_artifact_root() -> tuple[Path, dict]:
    candidates = [
        _secondary_manifest_explicit,
        Path(f"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
        Path(f"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("ARTIFACT_MANIFEST.json"))

    seen = set()
    for manifest_path in candidates:
        manifest_path = manifest_path.expanduser()
        if manifest_path in seen or not manifest_path.is_file():
            continue
        seen.add(manifest_path)
        try:
            info = json.loads(manifest_path.read_text())
        except Exception:
            continue
        sha256 = str(info.get("model", {}).get("weight_sha256", ""))
        if sha256 == _secondary_expected_sha256:
            return manifest_path.parent, info
    raise FileNotFoundError(
        "Could not find the independent-seed artifact with weight SHA256 "
        + _secondary_expected_sha256
    )


SECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()
SECONDARY_WEIGHTS_ROOT = WORKING_DIR / "secondary_seed_weights"
copy_or_extract_tree(
    SECONDARY_ARTIFACTS / "weights",
    SECONDARY_ARTIFACTS / "weights.zip",
    SECONDARY_WEIGHTS_ROOT,
)
SECONDARY_WEIGHTS_PATH = (
    SECONDARY_WEIGHTS_ROOT
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
SECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / "config.json"
for _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):
    if not _required_secondary_path.is_file():
        raise FileNotFoundError(f"Missing secondary model file: {_required_secondary_path}")


def _sha256_file(path: Path) -> str:
    digest = _hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)
if _secondary_actual_sha256 != _secondary_expected_sha256:
    raise RuntimeError(
        "Secondary model checksum mismatch: "
        f"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}"
    )

os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(SECONDARY_WEIGHTS_PATH)
os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"] = "0.15"
print("Secondary artifact:", SECONDARY_ARTIFACTS)
print("Secondary weight:", SECONDARY_WEIGHTS_PATH)
print("Secondary SHA256:", _secondary_actual_sha256)
print("Secondary edge-logit weight:", os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])

os.environ["BIOHUB_SECONDARY_DETECTION_WEIGHT"] = "0.475"
os.environ["BIOHUB_SECONDARY_LINK_MODE"] = "low_margin_consensus"
os.environ["BIOHUB_SECONDARY_MIX_TEMPERATURE"] = "1"
os.environ["BIOHUB_SECONDARY_LOW_MARGIN_MAX"] = "0.35"
os.environ["BIOHUB_DUAL_SEED_EDGE_THRESHOLD"] = "0.48"


In [ ]:
"""Validated Zh001r identity data for association-head training.

The public ``zh001r_nodes.npz`` stores one ``(N, 4)`` array per crop-frame with
columns ``[t, z, y, x]``.  Our registration sidecar stores row-aligned track and
parent-track identities as ``tid_{crop}_{frame}`` / ``pid_{crop}_{frame}``.

This module joins those assets without guessing, validates the complete schema,
and produces dense float32 transition matrices compatible with
``train_unet_transformer.compute_batch_loss``:

* continuation: ``source.track_id == target.track_id``;
* division daughter: ``source.track_id == target.parent_track_id``.

The sidecar is derived data.  Any missing row, duplicate identity, ambiguous
two-parent target, or row-count mismatch is therefore fatal rather than silently
being treated as an unlabelled node.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator

import numpy as np
import torch


N_FRAMES_PER_CROP = 20
EXPECTED_FULL_CROPS = 72
EXPECTED_CONTINUATION_LINKS = 1_192_441
EXPECTED_DIVISION_DAUGHTER_LINKS = 65_741

_NODE_RE = re.compile(r"^f(\d+)$")
_IDENT_RE = re.compile(r"^(tid|pid)_(\d+)_(\d+)$")


class EdgeDataError(ValueError):
    """The node pack and identity sidecar do not describe one trustworthy corpus."""


@dataclass(frozen=True)
class EdgeAssets:
    nodes: Path
    identity: Path


@dataclass(frozen=True)
class IdentityFrame:
    crop: int
    frame: int
    time: int
    coords: np.ndarray
    track_ids: np.ndarray
    parent_track_ids: np.ndarray


@dataclass(frozen=True)
class EdgePair:
    crop: int
    source_frame: int
    target_frame: int
    source_coords: np.ndarray
    target_coords: np.ndarray
    target: torch.Tensor
    continuation_links: int
    division_daughter_links: int


def _unique_named(search_roots: Iterable[str | Path], name: str) -> Path:
    hits: set[Path] = set()
    for raw_root in search_roots:
        root = Path(raw_root)
        if root.is_file():
            if root.name == name:
                hits.add(root.resolve())
            continue
        direct = root / name
        if direct.is_file():
            hits.add(direct.resolve())
        if root.is_dir():
            hits.update(p.resolve() for p in root.rglob(name) if p.is_file())
    if not hits:
        raise FileNotFoundError(f"could not find {name!r} under: "
                                + ", ".join(map(str, search_roots)))
    if len(hits) != 1:
        raise EdgeDataError(f"ambiguous {name!r}; found {len(hits)} copies: "
                            + ", ".join(map(str, sorted(hits))))
    return next(iter(hits))


def discover_edge_assets(
    search_roots: Iterable[str | Path],
    *,
    nodes_name: str = "zh001r_nodes.npz",
    identity_name: str = "zh001r_identity.npz",
) -> EdgeAssets:
    """Discover exactly one node pack and one identity sidecar.

    The two files may live in different Kaggle input mounts.  Ambiguity is an
    error because silently choosing an older sidecar would poison every target.
    """
    roots = tuple(search_roots)
    if not roots:
        raise ValueError("at least one search root is required")
    return EdgeAssets(
        nodes=_unique_named(roots, nodes_name),
        identity=_unique_named(roots, identity_name),
    )


def load_edge_data(
    search_roots: Iterable[str | Path],
    *,
    n_frames: int = N_FRAMES_PER_CROP,
    assert_published_totals: bool = True,
) -> "Zh001rEdgeData":
    """Discover both assets and return one validated edge-data corpus."""
    assets = discover_edge_assets(search_roots)
    return Zh001rEdgeData(
        assets.nodes,
        assets.identity,
        n_frames=n_frames,
        assert_published_totals=assert_published_totals,
    )


def _as_int_ids(value: np.ndarray, *, key: str, allow_negative: bool) -> np.ndarray:
    a = np.asarray(value)
    if a.ndim != 1:
        raise EdgeDataError(f"{key}: expected a 1-D identity array, got {a.shape}")
    if not np.issubdtype(a.dtype, np.number) or not np.isfinite(a).all():
        raise EdgeDataError(f"{key}: identities must be finite numbers")
    rounded = np.rint(a)
    if not np.array_equal(a, rounded):
        raise EdgeDataError(f"{key}: identities must be integer-valued")
    out = rounded.astype(np.int64, copy=False)
    if not allow_negative and np.any(out < 0):
        bad = int(np.sum(out < 0))
        raise EdgeDataError(f"{key}: {bad} nodes have no registered track identity")
    if allow_negative and np.any(out < -1):
        raise EdgeDataError(f"{key}: parent ids may use -1 only as the no-parent sentinel")
    return out


def build_transition_target(
    source_track_ids: np.ndarray,
    target_track_ids: np.ndarray,
    target_parent_track_ids: np.ndarray,
) -> tuple[torch.Tensor, int, int]:
    """Build one adjacent-frame target matrix and return its link-type counts."""
    rows, cols, n_continuation, n_division, shape = _transition_indices(
        source_track_ids, target_track_ids, target_parent_track_ids
    )
    target = torch.zeros(shape, dtype=torch.float32)
    if rows:
        target[rows, cols] = 1.0
    return target, n_continuation, n_division


def _transition_indices(
    source_track_ids: np.ndarray,
    target_track_ids: np.ndarray,
    target_parent_track_ids: np.ndarray,
) -> tuple[list[int], list[int], int, int, tuple[int, int]]:
    """Validate a pair and return sparse positive indices without allocating N x M."""
    src = _as_int_ids(source_track_ids, key="source track ids", allow_negative=False)
    tgt = _as_int_ids(target_track_ids, key="target track ids", allow_negative=False)
    pid = _as_int_ids(target_parent_track_ids, key="target parent ids", allow_negative=True)
    if len(tgt) != len(pid):
        raise EdgeDataError(
            f"target identity length mismatch: tid={len(tgt)} pid={len(pid)}"
        )
    if len(np.unique(src)) != len(src):
        raise EdgeDataError("duplicate track_id in source frame")
    if len(np.unique(tgt)) != len(tgt):
        raise EdgeDataError("duplicate track_id in target frame")

    source_row = {int(track_id): i for i, track_id in enumerate(src)}
    rows: list[int] = []
    cols: list[int] = []
    n_continuation = 0
    n_division = 0
    for j, (track_id, parent_id) in enumerate(zip(tgt, pid, strict=True)):
        cont_i = source_row.get(int(track_id))
        div_i = source_row.get(int(parent_id)) if parent_id >= 0 else None
        if cont_i is not None and div_i is not None:
            raise EdgeDataError(
                "target node has both a continuation parent and a division parent in "
                f"the source frame (track_id={int(track_id)}, parent_track_id={int(parent_id)})"
            )
        if cont_i is not None:
            rows.append(cont_i)
            cols.append(j)
            n_continuation += 1
        elif div_i is not None:
            rows.append(div_i)
            cols.append(j)
            n_division += 1
    return rows, cols, n_continuation, n_division, (len(src), len(tgt))


class Zh001rEdgeData:
    """Eagerly validated, row-aligned Zh001r node identities."""

    def __init__(
        self,
        nodes_path: str | Path,
        identity_path: str | Path,
        *,
        n_frames: int = N_FRAMES_PER_CROP,
        assert_published_totals: bool = True,
    ) -> None:
        self.nodes_path = Path(nodes_path)
        self.identity_path = Path(identity_path)
        self.n_frames = int(n_frames)
        if self.n_frames < 2:
            raise ValueError("n_frames must be at least 2")
        if not self.nodes_path.is_file():
            raise FileNotFoundError(self.nodes_path)
        if not self.identity_path.is_file():
            raise FileNotFoundError(self.identity_path)

        with np.load(self.nodes_path, allow_pickle=False) as node_pack, \
                np.load(self.identity_path, allow_pickle=False) as identity_pack:
            self._frames = self._load_frames(node_pack, identity_pack)

        crops = sorted({crop for crop, _ in self._frames})
        expected_crops = list(range(len(crops)))
        if crops != expected_crops:
            raise EdgeDataError(
                f"crop ids must be contiguous from zero; found {crops[:8]}"
            )
        self.crops = tuple(crops)
        self.n_crops = len(crops)
        self.counts = self.link_counts()
        is_full = (
            self.n_frames == N_FRAMES_PER_CROP
            and self.n_crops == EXPECTED_FULL_CROPS
            and len(self._frames) == EXPECTED_FULL_CROPS * N_FRAMES_PER_CROP
        )
        if assert_published_totals and is_full:
            observed = (
                self.counts["continuation_links"],
                self.counts["division_daughter_links"],
            )
            expected = (
                EXPECTED_CONTINUATION_LINKS,
                EXPECTED_DIVISION_DAUGHTER_LINKS,
            )
            if observed != expected:
                raise EdgeDataError(
                    "full Zh001r association totals disagree with the registered authority: "
                    f"observed continuation/division={observed}, expected={expected}"
                )

    def _load_frames(self, node_pack, identity_pack) -> dict[tuple[int, int], IdentityFrame]:
        node_indices: dict[int, str] = {}
        for key in node_pack.files:
            match = _NODE_RE.fullmatch(key)
            if match is None:
                raise EdgeDataError(f"unexpected node-pack key {key!r}; expected f<index>")
            index = int(match.group(1))
            if index in node_indices:
                raise EdgeDataError(f"duplicate node frame index {index}")
            node_indices[index] = key
        if not node_indices:
            raise EdgeDataError("node pack is empty")
        expected_indices = set(range(max(node_indices) + 1))
        if set(node_indices) != expected_indices:
            missing = sorted(expected_indices - set(node_indices))
            raise EdgeDataError(f"node frame indices are not contiguous; missing {missing[:8]}")
        if len(node_indices) % self.n_frames:
            raise EdgeDataError(
                f"{len(node_indices)} node frames is not divisible by {self.n_frames}"
            )

        identity_keys: dict[tuple[str, int, int], str] = {}
        for key in identity_pack.files:
            match = _IDENT_RE.fullmatch(key)
            if match is None:
                raise EdgeDataError(
                    f"unexpected identity key {key!r}; expected tid_<crop>_<frame> or pid_<crop>_<frame>"
                )
            kind, crop_text, frame_text = match.groups()
            ident = (kind, int(crop_text), int(frame_text))
            if ident in identity_keys:
                raise EdgeDataError(f"duplicate identity key {key!r}")
            identity_keys[ident] = key

        n_crops = len(node_indices) // self.n_frames
        required = {
            (kind, crop, frame)
            for crop in range(n_crops)
            for frame in range(self.n_frames)
            for kind in ("tid", "pid")
        }
        actual = set(identity_keys)
        if actual != required:
            missing = sorted(required - actual)
            extra = sorted(actual - required)
            raise EdgeDataError(
                "identity schema does not exactly cover the node pack; "
                f"missing={missing[:6]} extra={extra[:6]}"
            )

        frames: dict[tuple[int, int], IdentityFrame] = {}
        for crop in range(n_crops):
            previous_time: int | None = None
            for frame in range(self.n_frames):
                node_key = node_indices[crop * self.n_frames + frame]
                nodes = np.asarray(node_pack[node_key])
                if nodes.ndim != 2 or nodes.shape[1] != 4:
                    raise EdgeDataError(f"{node_key}: expected (N, 4), got {nodes.shape}")
                if not np.issubdtype(nodes.dtype, np.number) or not np.isfinite(nodes).all():
                    raise EdgeDataError(f"{node_key}: node rows must be finite numbers")
                times = nodes[:, 0]
                if len(times):
                    rounded_times = np.rint(times)
                    if not np.array_equal(times, rounded_times) or len(np.unique(rounded_times)) != 1:
                        raise EdgeDataError(f"{node_key}: all rows must carry one integer time")
                    time = int(rounded_times[0])
                else:
                    time = frame if previous_time is None else previous_time + 1
                if previous_time is not None and time != previous_time + 1:
                    raise EdgeDataError(
                        f"crop {crop}: node times are not adjacent at frame {frame}: "
                        f"{previous_time} -> {time}"
                    )
                previous_time = time

                tid_key = identity_keys[("tid", crop, frame)]
                pid_key = identity_keys[("pid", crop, frame)]
                tids = _as_int_ids(identity_pack[tid_key], key=tid_key, allow_negative=False)
                pids = _as_int_ids(identity_pack[pid_key], key=pid_key, allow_negative=True)
                if len(nodes) != len(tids) or len(nodes) != len(pids):
                    raise EdgeDataError(
                        f"crop {crop} frame {frame}: row mismatch nodes={len(nodes)} "
                        f"tid={len(tids)} pid={len(pids)}"
                    )
                if len(np.unique(tids)) != len(tids):
                    raise EdgeDataError(f"crop {crop} frame {frame}: duplicate track_id")
                frames[(crop, frame)] = IdentityFrame(
                    crop=crop,
                    frame=frame,
                    time=time,
                    coords=np.asarray(nodes[:, 1:], dtype=np.float32),
                    track_ids=tids.copy(),
                    parent_track_ids=pids.copy(),
                )
        return frames

    def frame(self, crop: int, frame: int) -> IdentityFrame:
        try:
            return self._frames[(int(crop), int(frame))]
        except KeyError as exc:
            raise IndexError(f"unknown crop/frame ({crop}, {frame})") from exc

    def pair(self, crop: int, source_frame: int) -> EdgePair:
        if not 0 <= source_frame < self.n_frames - 1:
            raise IndexError(
                f"source_frame must be in [0, {self.n_frames - 1}), got {source_frame}"
            )
        source = self.frame(crop, source_frame)
        target_frame = self.frame(crop, source_frame + 1)
        matrix, n_cont, n_div = build_transition_target(
            source.track_ids,
            target_frame.track_ids,
            target_frame.parent_track_ids,
        )
        return EdgePair(
            crop=int(crop),
            source_frame=int(source_frame),
            target_frame=int(source_frame + 1),
            source_coords=source.coords,
            target_coords=target_frame.coords,
            target=matrix,
            continuation_links=n_cont,
            division_daughter_links=n_div,
        )

    def iter_pairs(self, crops: Iterable[int] | None = None) -> Iterator[EdgePair]:
        selected = self.crops if crops is None else tuple(int(c) for c in crops)
        for crop in selected:
            if crop not in self.crops:
                raise IndexError(f"unknown crop {crop}")
            for frame in range(self.n_frames - 1):
                yield self.pair(crop, frame)

    def link_counts(self) -> dict[str, int]:
        continuation = division = 0
        for crop in sorted({c for c, _ in self._frames}):
            for frame in range(self.n_frames - 1):
                source = self._frames[(crop, frame)]
                target = self._frames[(crop, frame + 1)]
                _, _, n_cont, n_div, _ = _transition_indices(
                    source.track_ids, target.track_ids, target.parent_track_ids
                )
                continuation += n_cont
                division += n_div
        return {
            "continuation_links": continuation,
            "division_daughter_links": division,
            "association_links": continuation + division,
        }


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--root", action="append", required=True,
                        help="search root; repeat when nodes and identity use separate mounts")
    parser.add_argument("--no-published-assert", action="store_true",
                        help="skip the 72-crop published-total assertion")
    args = parser.parse_args(argv)
    assets = discover_edge_assets(args.root)
    data = Zh001rEdgeData(
        assets.nodes,
        assets.identity,
        assert_published_totals=not args.no_published_assert,
    )
    report = {
        "nodes": str(assets.nodes),
        "identity": str(assets.identity),
        "crops": data.n_crops,
        "frames_per_crop": data.n_frames,
        **data.counts,
    }
    print(json.dumps(report, indent=2, sort_keys=True))
    return 0


if os.environ.get("H1R_KERNEL") == "1":
    # Kaggle factory embeds this module as a notebook cell. Register that cell's
    # namespace so the following h1r_edge_train cell can use its normal import.
    sys.modules.setdefault("h1r_edge_data", sys.modules[__name__])
elif __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
"""Runnable S5 Zh001r association/appearance trainer.

Uses registered identities from :mod:`h1r_edge_data` and the vendored
``UNetNodeTransformer`` primitives.  Detection is never supervised here: the
public detection head is frozen byte-for-byte while the UNet representation and
association transformer train on adjacent two-frame windows.
"""
from __future__ import annotations

import argparse
import json
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from h1r_edge_data import Zh001rEdgeData, build_transition_target, discover_edge_assets


DEPLOY_DOWNSAMPLE = (1.0, 4.0, 4.0)
DEFAULT_NODE_CAP = 256


def epoch_sample_indices(n: int, cap: int, *, seed: int, epoch: int,
                         crop: int, frame: int) -> np.ndarray:
    """Deterministic within an epoch, deliberately different between epochs."""
    if n <= cap:
        return np.arange(n, dtype=np.int64)
    ss = np.random.SeedSequence([int(seed), int(epoch), int(crop), int(frame)])
    return np.sort(np.random.default_rng(ss).choice(n, cap, replace=False)).astype(np.int64)


class Zh001rEdgeWindows(Dataset):
    """Adjacent two-frame images, capped identities, and row-aligned targets."""

    def __init__(self, data: Zh001rEdgeData, iso_path: str | Path,
                 crop_ids: Iterable[int], *, node_cap: int = DEFAULT_NODE_CAP,
                 seed: int = 0):
        self.data = data
        self.iso = np.load(Path(iso_path), mmap_mode="r")
        expected = (data.n_crops, data.n_frames)
        if self.iso.ndim != 5 or tuple(self.iso.shape[:2]) != expected:
            raise ValueError(f"image/node corpus mismatch: images={self.iso.shape[:2]} nodes={expected}")
        self.crop_ids = tuple(sorted(int(c) for c in crop_ids))
        if not self.crop_ids or any(c not in data.crops for c in self.crop_ids):
            raise ValueError(f"invalid/empty crop selection: {self.crop_ids}")
        self.node_cap = int(node_cap)
        if self.node_cap < 1:
            raise ValueError("node_cap must be positive")
        self.seed = int(seed)
        self.epoch = 0
        self.items = [(c, f) for c in self.crop_ids for f in range(data.n_frames - 1)]
        self._quantiles: dict[int, tuple[float, float]] = {}

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __len__(self) -> int:
        return len(self.items)

    def _select(self, crop: int, frame: int,
                required: set[int] | None = None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Sample up to ``node_cap`` nodes, always retaining ``required`` track ids.

        Source and target frames used to be drawn INDEPENDENTLY. With ~942 nuclei per frame
        and a cap of 256 that left a target's true parent in the source draw only ~27% of the
        time, so ~73% of columns looked like "no parent" while deployment has ~92% *with* one
        -- a background prior about 9x off, on the exact quantity the parental softmax has to
        calibrate. Retaining the required parents removes that bias at no memory cost: the cap
        is unchanged, only which nodes fill it.
        """
        fr = self.data.frame(crop, frame)
        n = len(fr.track_ids)
        take = epoch_sample_indices(n, self.node_cap, seed=self.seed,
                                    epoch=self.epoch, crop=crop, frame=frame)
        if required and n > self.node_cap:
            ids = np.asarray(fr.track_ids)
            need = np.flatnonzero(np.isin(ids, np.fromiter(required, dtype=ids.dtype,
                                                           count=len(required))))
            if need.size:
                keep = np.union1d(take, need)
                if keep.size > self.node_cap:
                    # Required parents win the cap; drop the tail of the random fill.
                    filler = np.setdiff1d(keep, need, assume_unique=False)
                    room = max(0, self.node_cap - need.size)
                    keep = np.union1d(need[: self.node_cap], filler[:room])
                take = np.sort(keep).astype(np.int64)
        return fr.coords[take], fr.track_ids[take], fr.parent_track_ids[take]

    def __getitem__(self, index: int) -> dict:
        crop, frame = self.items[index]
        if crop not in self._quantiles:
            whole = np.asarray(self.iso[crop], dtype=np.float32)
            self._quantiles[crop] = (float(np.quantile(whole, 0.001)),
                                     float(np.quantile(whole, 0.999)))
        qlo, qhi = self._quantiles[crop]
        raw = np.asarray(self.iso[crop, frame:frame + 2], dtype=np.float32)
        imgs = np.clip((raw - qlo) / (qhi - qlo + 1e-6), 0.0, None)
        c1, tid1, pid1 = self._select(crop, frame + 1)
        # Draw the TARGET frame first, then retain those targets' true parents in the source
        # draw, so the sampled matrix keeps the positives it is supposed to teach.
        required_parents = {int(t) for t in tid1}
        required_parents |= {int(p) for p in pid1 if int(p) >= 0}
        c0, tid0, _ = self._select(crop, frame, required=required_parents)
        target, _, _ = build_transition_target(tid0, tid1, pid1)
        # `target` already encodes divisions: _transition_indices resolves a daughter to
        # its MOTHER's row via `div_i = source_row.get(parent_id)`.  Supervising it as-is
        # is the whole point of this lane -- the 65,741 division-daughter links are the
        # asset the field cannot fork.
        #
        # The real hazard is SAMPLING, not divisions.  `source_row` is built from the
        # sampled source subset, so if the node cap drops a column's true parent the
        # column has no positive and supervising it as background is a FALSE NEGATIVE.
        # That applies identically to continuations and divisions, so it is masked on the
        # sampling condition alone.
        full_src = set(map(int, self.data.frame(crop, frame).track_ids))
        sampled_src = set(map(int, tid0))
        unresolved_cols = np.array([
            ((int(t) in full_src) or (int(p) >= 0 and int(p) in full_src))
            and not ((int(t) in sampled_src) or (int(p) >= 0 and int(p) in sampled_src))
            for t, p in zip(tid1, pid1, strict=True)
        ], dtype=bool)
        # Division columns that ARE resolvable in this sample, kept for loss upweighting.
        division_cols = np.array([
            int(p) >= 0 and int(p) in sampled_src and int(t) not in sampled_src
            for t, p in zip(tid1, pid1, strict=True)
        ], dtype=bool)
        return {
            "crop": crop, "frame": frame, "imgs": torch.from_numpy(imgs),
            "coords0": torch.from_numpy(c0.copy()), "coords1": torch.from_numpy(c1.copy()),
            "tid0": torch.from_numpy(tid0.copy()), "tid1": torch.from_numpy(tid1.copy()),
            "target": target, "division_cols": torch.from_numpy(division_cols),
            "unresolved_cols": torch.from_numpy(unresolved_cols),
        }


def collate_edge_windows(rows: list[dict]) -> dict:
    b = len(rows); m0 = max(len(r["tid0"]) for r in rows); m1 = max(len(r["tid1"]) for r in rows)
    coords0 = torch.zeros(b, m0, 3); coords1 = torch.zeros(b, m1, 3)
    mask0 = torch.zeros(b, m0, dtype=torch.bool); mask1 = torch.zeros(b, m1, dtype=torch.bool)
    tid0 = torch.full((b, m0), -1, dtype=torch.long); tid1 = torch.full((b, m1), -1, dtype=torch.long)
    targets = torch.zeros(b, m0, m1); div_cols = torch.zeros(b, m1, dtype=torch.bool)
    unres_cols = torch.zeros(b, m1, dtype=torch.bool)
    for i, r in enumerate(rows):
        n0, n1 = len(r["tid0"]), len(r["tid1"])
        coords0[i, :n0] = r["coords0"]; coords1[i, :n1] = r["coords1"]
        mask0[i, :n0] = True; mask1[i, :n1] = True
        tid0[i, :n0] = r["tid0"]; tid1[i, :n1] = r["tid1"]
        targets[i, :n0, :n1] = r["target"]; div_cols[i, :n1] = r["division_cols"]
        unres_cols[i, :n1] = r["unresolved_cols"]
    return {"imgs": torch.stack([r["imgs"] for r in rows]), "coords0": coords0,
            "coords1": coords1, "mask0": mask0, "mask1": mask1, "tid0": tid0,
            "tid1": tid1, "targets": targets, "division_cols": div_cols,
            "unresolved_cols": unres_cols,
            "crop": torch.tensor([r["crop"] for r in rows]),
            "frame": torch.tensor([r["frame"] for r in rows])}


class EdgeTrainingModel(nn.Module):
    """Public full model plus an optional auxiliary appearance projector."""

    def __init__(self, base: nn.Module, appearance_dim: int = 0):
        super().__init__(); self.base = base
        channels = int(base.unet_out_channels)
        self.projector = (nn.Linear(channels, appearance_dim) if appearance_dim > 0 else None)
        for p in self.base.detect_head.parameters():
            p.requires_grad_(False)
        self.trunk_mode = TRUNK_MODE
        if self.trunk_mode == "frozen":
            # Freeze the SHARED TRUNK too. detect_head's bytes being frozen does not freeze
            # detection: every detection logit is detect_head(unet(x)), and `unet_out` is also
            # the exact tensor deployment feeds to predict_edges. Training the trunk therefore
            # rewrites every edge feature under a transformer trained on the old representation
            # -- and that representation IS the 0.915 substrate. Freezing it is the honest
            # default and the baseline any adaptive mode must beat.
            for p in self.base.unet.parameters():
                p.requires_grad_(False)


def load_public_full_model(T, weights: str | Path, device: torch.device,
                           *, appearance_dim: int = 0) -> EdgeTrainingModel:
    """Reconstruct and strictly load the exact full public checkpoint."""
    weights = Path(weights); cfg_path = weights.parent / "config.json"
    if not cfg_path.is_file():
        raise FileNotFoundError(f"public config missing next to weights: {cfg_path}")
    cfg = json.loads(cfg_path.read_text())
    out_c = int(cfg["unet_out_channels"]); layers = list(cfg["unet_layers"])
    unet = T.TemporalUNet3D(in_channels=1, out_channels=out_c, layers=layers)
    base = T.UNetNodeTransformer(unet=unet, unet_out_channels=out_c,
                                 pos_feat_dim=4 * T._POS_EMBED_DIM)
    state = torch.load(weights, map_location="cpu", weights_only=True)
    if not isinstance(state, dict) or not state or not all(torch.is_tensor(v) for v in state.values()):
        raise ValueError("initial weights must be a bare full-model state_dict")
    base.load_state_dict(state, strict=True)
    return EdgeTrainingModel(base, appearance_dim).to(device)


DIVISION_LOSS_WEIGHT = float(os.environ.get("H1R_DIV_WEIGHT", "3.0"))

# The trunk contract, chosen EXPLICITLY rather than implied by a docstring.
#   frozen : UNet + detect_head frozen. Only transformer (+ projector) train. Zero
#            representation drift by construction. DEFAULT, and the baseline the other
#            modes must beat.
#   adapt  : UNet trains. Detection behaviour WILL move even though detect_head's bytes do
#            not. Nothing calls this "frozen detection".
#   distill: UNet trains, with a frozen teacher penalising association-logit drift.
TRUNK_MODE = os.environ.get("H1R_TRUNK_MODE", "frozen").strip().lower()
if TRUNK_MODE not in {"frozen", "adapt", "distill"}:
    raise ValueError(f"H1R_TRUNK_MODE must be frozen|adapt|distill, got {TRUNK_MODE!r}")


@torch.no_grad()
def detection_drift(model: "EdgeTrainingModel", reference: torch.Tensor,
                    imgs: torch.Tensor) -> dict[str, float]:
    """Measure how far detection logits have moved from a frozen reference.

    The previous guard compared ``detect_head``'s state_dict bytes to their initial values.
    That is TAUTOLOGICAL: those parameters were excluded from the optimizer, so they cannot
    change, and the check proves nothing about detection BEHAVIOUR. What actually moves is
    the shared trunk feeding them.

    This is the instrument for the cheap pre-flight gate: run a short adapt-mode fine-tune
    and read ``max_abs``. If detection barely moves, the whole drift apparatus is
    unnecessary and should not be built.
    """
    was_training = model.training
    model.eval()
    # encode returns (unet_out, det_logits) where det_logits is a LIST of W tensors.
    _, logits = model.base.encode(imgs)
    logits = torch.stack(logits, dim=1) if isinstance(logits, (list, tuple)) else logits
    model.train(was_training)
    delta = (logits.float() - reference.float())
    denom = reference.float().abs().max().clamp_min(1e-6)
    return {"max_abs": float(delta.abs().max()),
            "mean_abs": float(delta.abs().mean()),
            "relative": float(delta.abs().max() / denom)}


def _selection_score(link_top1: float, candidate_recall: float) -> float:
    """Harmonic mean, so a checkpoint cannot win on recall alone.

    The previous criterion was ``link_top1 * candidate_recall`` -- the exact product this
    repo already rejected once, in ``h1r_trainer_patch.py:19-22``: *"the vendored
    test_acc * test_recall ... has no precision term and selects the junkiest detector"*.
    A model that spreads probability mass over every column scores well on that product.
    """
    if link_top1 <= 0.0 or candidate_recall <= 0.0:
        return 0.0
    return 2.0 * link_top1 * candidate_recall / (link_top1 + candidate_recall)


def continuation_loss(logits: torch.Tensor, target: torch.Tensor,
                      source_mask: torch.Tensor, target_mask: torch.Tensor,
                      division_cols: torch.Tensor,
                      unresolved_cols: torch.Tensor | None = None) -> torch.Tensor:
    """Parental-softmax focal BCE over association columns.

    Divisions ARE supervised and upweighted by ``H1R_DIV_WEIGHT`` (default 3.0, matching
    this repo's own P3 trainer patch; Trackastra uses 11).  `target` already resolves a
    daughter to her mother's row, so no extra construction is needed.

    Only UNRESOLVABLE columns are dropped -- those whose true parent exists in the frame
    but was not sampled by the node cap.  Supervising those as background would be a false
    negative.  That is a sampling artifact and applies to continuations and divisions
    alike, so it is masked on the sampling condition, never on "is this a division".

    The body runs in fp32 with autocast DISABLED: ``F.binary_cross_entropy`` is banned
    under CUDA autocast (documented in ``h1r_trainer_patch.py:12-13``) and raises at
    runtime -- a CPU smoke cannot reproduce it because CPU autocast permits the op.
    """
    losses = []
    for b in range(logits.shape[0]):
        ns = int(source_mask[b].sum()); nt = int(target_mask[b].sum())
        if not ns or not nt:
            continue
        z = logits[b, :ns, :nt]; y = target[b, :ns, :nt]
        if unresolved_cols is None:
            keep = torch.ones(nt, dtype=torch.bool, device=z.device)
        else:
            keep = ~unresolved_cols[b, :nt]
        if not bool(keep.any()):
            continue
        col_w = torch.where(division_cols[b, :nt][keep],
                            z.new_full((), DIVISION_LOSS_WEIGHT), z.new_ones(()))
        z, y = z[:, keep], y[:, keep]
        bg = torch.zeros(1, z.shape[1], device=z.device, dtype=z.dtype)
        prob = torch.softmax(torch.cat([z, bg], 0), 0)[:-1].clamp(1e-7, 1 - 1e-7)
        # Match the vendored sparse-lineage semantics: supervise cells touching an
        # annotated continuation row or column, not the entire N x M sea of unknowns.
        active_rows = y.sum(1) > 0
        active_cols = y.sum(0) > 0
        supervised = active_rows.unsqueeze(1) | active_cols.unsqueeze(0)
        if not bool(supervised.any()):
            continue
        # Explicit fp32 BCE on the clamped probabilities: identical to
        # torch.nn.functional binary cross-entropy with reduction none, but never touches the op that CUDA
        # autocast bans (DG-001), so the guard above is belt AND braces.
        bce = -(y * torch.log(prob) + (1.0 - y) * torch.log(1.0 - prob))
        p_t = prob * y + (1.0 - prob) * (1.0 - y)
        focal = ((1.0 - p_t) ** 2) * bce * col_w.unsqueeze(0)
        losses.append(focal[supervised].mean())
    return torch.stack(losses).mean() if losses else logits.sum() * 0.0


def appearance_triplet_loss(projector: nn.Module | None, feat0: torch.Tensor,
                            feat1: torch.Tensor, tid0: torch.Tensor, tid1: torch.Tensor,
                            mask0: torch.Tensor, mask1: torch.Tensor,
                            margin: float = 0.2) -> torch.Tensor:
    """Hard-negative temporal identity triplets; no identity match means zero loss."""
    if projector is None:
        return feat0.sum() * 0.0
    losses = []
    e0 = F.normalize(projector(feat0.float()), dim=-1)
    e1 = F.normalize(projector(feat1.float()), dim=-1)
    for b in range(feat0.shape[0]):
        n0, n1 = int(mask0[b].sum()), int(mask1[b].sum())
        ids0 = tid0[b, :n0]; ids1 = tid1[b, :n1]
        lookup = {int(v): i for i, v in enumerate(ids0.tolist())}
        for j, identity in enumerate(ids1.tolist()):
            i = lookup.get(int(identity))
            if i is None or n0 < 2:
                continue
            negative = torch.arange(n0, device=feat0.device) != i
            distances = 1.0 - e1[b, j].unsqueeze(0) @ e0[b, :n0].T
            k = torch.arange(n0, device=feat0.device)[negative][distances[0, negative].argmin()]
            losses.append(F.triplet_margin_loss(e1[b, j:j + 1], e0[b, i:i + 1],
                                                 e0[b, k:k + 1], margin=margin))
    return torch.stack(losses).mean() if losses else (e0.sum() + e1.sum()) * 0.0


def forward_batch(T, model: EdgeTrainingModel, batch: dict, device: torch.device,
                  *, triplet_weight: float = 0.0) -> tuple[torch.Tensor, dict]:
    imgs = batch["imgs"].to(device, dtype=torch.float32)
    c0 = batch["coords0"].to(device); c1 = batch["coords1"].to(device)
    m0 = batch["mask0"].to(device); m1 = batch["mask1"].to(device)
    target = batch["targets"].to(device); div = batch["division_cols"].to(device)
    unresolved = batch["unresolved_cols"].to(device)
    tid0 = batch["tid0"].to(device); tid1 = batch["tid1"].to(device)
    # Bypass detect_head completely: this lane trains association representations only.
    maps = model.base.unet(imgs.unsqueeze(2))
    f0 = model.base._index_features(maps[:, 0], c0, m0)
    f1 = model.base._index_features(maps[:, 1], c1, m1)
    scale = torch.tensor(DEPLOY_DOWNSAMPLE, device=device)
    pc0, pc1 = c0 * scale, c1 * scale
    shape = (2, imgs.shape[2], imgs.shape[3] * 4, imgs.shape[4] * 4)
    t0 = torch.zeros((*pc0.shape[:-1], 1), device=device)
    t1 = torch.ones((*pc1.shape[:-1], 1), device=device)
    p0 = T._pos_embed_torch(torch.cat([t0, pc0], -1), shape)
    p1 = T._pos_embed_torch(torch.cat([t1, pc1], -1), shape)
    logits = model.base.predict_edges(f0, f1, pc0, pc1, p0, p1, m0, m1)
    # F.binary_cross_entropy is BANNED under CUDA autocast and raises at runtime; the CPU
    # autocast path permits it, so no CPU smoke can catch this. Force fp32 with autocast
    # disabled for the loss only -- the UNet/transformer forward above keeps its fp16.
    with torch.autocast(device.type, enabled=False):
        assoc = continuation_loss(logits.float(), target, m0, m1, div, unresolved)
    triplet = (appearance_triplet_loss(model.projector, f0, f1, tid0, tid1, m0, m1)
               if triplet_weight > 0.0 else f0.sum() * 0.0)
    loss = assoc + float(triplet_weight) * triplet
    return loss, {"logits": logits, "association_loss": assoc, "triplet_loss": triplet}


@torch.no_grad()
def evaluate(T, model: EdgeTrainingModel, loader: DataLoader, device: torch.device,
             *, candidate_threshold: float = 0.48) -> dict:
    model.eval(); correct = total = reached = 0; losses = []
    for batch in loader:
        loss, out = forward_batch(T, model, batch, device)
        losses.append(float(loss)); logits = out["logits"]
        targets = batch["targets"].to(device); div = batch["division_cols"].to(device)
        for b in range(logits.shape[0]):
            ns, nt = int(batch["mask0"][b].sum()), int(batch["mask1"][b].sum())
            z = logits[b, :ns, :nt]; y = targets[b, :ns, :nt]
            bg = torch.zeros(1, nt, device=device, dtype=z.dtype)
            probs = torch.softmax(torch.cat([z, bg], 0), 0)[:-1]
            for j in range(nt):
                if bool(div[b, j]) or not bool(y[:, j].any()):
                    continue
                i = int(y[:, j].argmax()); total += 1
                correct += int(int(probs[:, j].argmax()) == i)
                reached += int(float(probs[i, j]) >= candidate_threshold)
    return {"loss": float(np.mean(losses)) if losses else float("nan"),
            "link_top1": correct / max(total, 1),
            "candidate_recall": reached / max(total, 1),
            "select_score": _selection_score(correct / max(total, 1),
                                             reached / max(total, 1)),
            "n_links": total}


def _rng_state() -> dict:
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def _discover_resume(out_dir: Path) -> Path:
    """Find a resume checkpoint, including one attached as a Kaggle dataset.

    ``/kaggle/working`` starts EMPTY on every batch run, so ``<out>/edge_resume.pth`` never
    exists on a fresh push and ``--resume`` was dead code that read as a safety net. A prior
    run's state can only arrive as an ATTACHED INPUT, so search there too.
    """
    local = out_dir / "edge_resume.pth"
    if local.is_file():
        return local
    root = Path("/kaggle/input")
    if root.is_dir():
        found = sorted(root.glob("*/edge_resume.pth")) + sorted(root.glob("*/*/edge_resume.pth"))
        if found:
            return found[0]
    return local


def save_resume(path: str | Path, model: EdgeTrainingModel, optimizer, scheduler, scaler,
                *, epoch: int, best: float, history: list[dict]) -> None:
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "epoch": int(epoch), "best": float(best), "history": history,
                "rng": _rng_state()}, tmp)
    tmp.replace(path)


def load_resume(path: str | Path, model: EdgeTrainingModel, optimizer, scheduler, scaler,
                device: torch.device) -> tuple[int, float, list[dict]]:
    ck = torch.load(path, map_location=device, weights_only=False)
    for key in ("model", "optimizer", "scheduler", "scaler", "epoch", "best", "history", "rng"):
        if key not in ck: raise ValueError(f"resume checkpoint missing {key}")
    model.load_state_dict(ck["model"], strict=True); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    random.setstate(ck["rng"]["python"]); np.random.set_state(ck["rng"]["numpy"])
    torch.set_rng_state(ck["rng"]["torch"])
    if torch.cuda.is_available() and ck["rng"]["cuda"]:
        torch.cuda.set_rng_state_all(ck["rng"]["cuda"])
    return int(ck["epoch"]) + 1, float(ck["best"]), list(ck["history"])


def run(args) -> dict:
    trainer_dir = Path(args.trainer_dir)
    # The vendored trainer imports biohub_tracking from its sibling src tree.
    # Notebook execution must recreate the PYTHONPATH used by the CLI scripts.
    sys.path.insert(0, str(trainer_dir.parent / "src"))
    sys.path.insert(0, str(trainer_dir))
    import train_unet_transformer as T
    assets = discover_edge_assets(args.root)
    data = Zh001rEdgeData(assets.nodes, assets.identity)
    iso = next((p for r in args.root for p in Path(r).rglob("zh001r_iso.npy")), None)
    if iso is None: raise FileNotFoundError("zh001r_iso.npy not found")
    val_ids = [c for c in data.crops if c % args.val_mod == 0]
    train_ids = [c for c in data.crops if c % args.val_mod != 0]
    train_ds = Zh001rEdgeWindows(data, iso, train_ids, node_cap=args.node_cap, seed=args.seed)
    val_ds = Zh001rEdgeWindows(data, iso, val_ids, node_cap=args.node_cap, seed=args.seed)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              collate_fn=collate_edge_windows)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                            collate_fn=collate_edge_windows)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(args.seed); random.seed(args.seed); np.random.seed(args.seed)
    model = load_public_full_model(T, args.weights, device, appearance_dim=args.appearance_dim)
    # Fixed probe batch + frozen reference logits: this is what makes detection drift
    # OBSERVABLE. Comparing detect_head's bytes (as the old guard did) is tautological --
    # those parameters are not in the optimizer and cannot move.
    _probe = next(iter(val_loader))
    _probe_imgs = _probe["imgs"].to(device)
    with torch.no_grad():
        model.eval(); _, _ref_logits = model.base.encode(_probe_imgs); model.train()
    _detect_reference = (torch.stack(_ref_logits, dim=1)
                         if isinstance(_ref_logits, (list, tuple)) else _ref_logits).detach().clone()
    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        raise ValueError("no trainable parameters: check H1R_TRUNK_MODE and appearance_dim")
    opt = torch.optim.AdamW(params, lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(args.epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    start, best, history = 0, -1.0, []
    resume = _discover_resume(Path(args.out))
    Path(args.out).mkdir(parents=True, exist_ok=True)
    public_config = Path(args.weights).parent / "config.json"
    (Path(args.out) / "config.json").write_text(public_config.read_text())
    if args.resume and resume.exists(): start, best, history = load_resume(resume, model, opt, sched, scaler, device)
    if not history:
        val_ds.set_epoch(0)
        zero_val = evaluate(T, model, val_loader, device)
        history.append({"epoch": -1, "val": zero_val})
        best = zero_val["select_score"]
        torch.save(model.base.state_dict(), Path(args.out) / "edge_predictor_best.pth")
        if model.projector is not None:
            torch.save(model.projector.state_dict(), Path(args.out) / "appearance_projector_best.pth")
        (Path(args.out) / "metrics.json").write_text(json.dumps(history, indent=1))
    for epoch in range(start, args.epochs):
        train_ds.set_epoch(epoch); model.train(); vals = []
        for step, batch in enumerate(train_loader):
            if args.max_steps and step >= args.max_steps: break
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16,
                                enabled=args.amp and device.type == "cuda"):
                loss, _ = forward_batch(T, model, batch, device, triplet_weight=args.triplet_weight)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
            vals.append(float(loss.detach()))
        # Validation keeps the epoch-0 cap sample fixed so checkpoint metrics are paired.
        sched.step(); metrics = evaluate(T, model, val_loader, device)
        row = {"epoch": epoch, "train_loss": float(np.mean(vals)), "val": metrics}; history.append(row)
        # Detection drift is REPORTED every epoch, not asserted away. In frozen mode it must
        # be identically zero; in adapt/distill it is the number that decides whether the
        # drift apparatus is worth building at all.
        drift = detection_drift(model, _detect_reference, _probe_imgs)
        row["detection_drift"] = drift
        if TRUNK_MODE == "frozen" and drift["max_abs"] > 1e-4:
            raise RuntimeError(
                f"H1R_TRUNK_MODE=frozen but detection logits moved by {drift['max_abs']:.3e} -- "
                "the trunk is not actually frozen"
            )
        score = metrics["select_score"]
        if score >= best:
            best = score; torch.save(model.base.state_dict(), Path(args.out) / "edge_predictor_best.pth")
            if model.projector is not None: torch.save(model.projector.state_dict(), Path(args.out) / "appearance_projector_best.pth")
        save_resume(resume, model, opt, sched, scaler, epoch=epoch, best=best, history=history)
        (Path(args.out) / "metrics.json").write_text(json.dumps(history, indent=1))
    return {"best": best, "history": history, "trunk_mode": TRUNK_MODE,
            "final_detection_drift": history[-1].get("detection_drift") if history else None}


def main(argv=None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--trainer-dir", required=True); p.add_argument("--root", action="append", required=True)
    p.add_argument("--weights", required=True); p.add_argument("--out", required=True)
    p.add_argument("--epochs", type=int, default=20); p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--node-cap", type=int, default=256); p.add_argument("--val-mod", type=int, default=6)
    p.add_argument("--seed", type=int, default=0); p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--max-steps", type=int, default=0); p.add_argument("--appearance-dim", type=int, default=64)
    p.add_argument("--triplet-weight", type=float, default=0.05); p.add_argument("--resume", action=argparse.BooleanOptionalAction, default=True)
    p.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)
    return 0 if run(p.parse_args(argv)) else 1


if __name__ == "__main__" and os.environ.get("H1R_KERNEL") != "1":
    raise SystemExit(main())


In [ ]:
"""Final cell for the H1-R S5 association-training kernels.

The preceding notebook cells embed h1r_edge_data and h1r_edge_train. This cell
verifies both Zh001r mounts, resolves the public full-model checkpoint, runs the
configured training arm, and publishes only training artifacts. It never submits.
"""
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path
from types import SimpleNamespace

import torch


if not torch.cuda.is_available():
    raise RuntimeError("H1-R S5 requires a Kaggle GPU; refusing a silent CPU run")
print("CUDA:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

edge_assets = discover_edge_assets([Path("/kaggle/input")])
iso_hits = sorted(Path("/kaggle/input").rglob("zh001r_iso.npy"))
if len(iso_hits) != 1:
    raise RuntimeError(f"expected exactly one zh001r_iso.npy, got {iso_hits}")
print("S5_ASSETS", json.dumps({
    "nodes": str(edge_assets.nodes),
    "identity": str(edge_assets.identity),
    "iso": str(iso_hits[0]),
}, sort_keys=True))

trainer_path = REPO_DIR / "scripts" / "train_unet_transformer.py"
init_weights = (
    REPO_DIR / "weights" / "unet_transformer" / "split_0" /
    "edge_predictor_best.pth"
)
for required in (trainer_path, init_weights):
    if not required.is_file():
        raise FileNotFoundError(required)

env = os.environ.get
out = Path("/kaggle/working/h1r_edge")
args = SimpleNamespace(
    trainer_dir=str(trainer_path.parent),
    root=["/kaggle/input"],
    weights=str(init_weights),
    out=str(out),
    epochs=int(env("H1R_EDGE_EPOCHS", "20")),
    batch_size=int(env("H1R_EDGE_BS", "1")),
    node_cap=int(env("H1R_EDGE_NODE_CAP", "256")),
    val_mod=int(env("H1R_EDGE_VAL_MOD", "6")),
    seed=int(env("H1R_EDGE_SEED", "0")),
    lr=float(env("H1R_EDGE_LR", "1e-4")),
    max_steps=int(env("H1R_EDGE_MAX_STEPS", "0")),
    appearance_dim=int(env("H1R_APPEARANCE_DIM", "64")),
    triplet_weight=float(env("H1R_TRIPLET_WEIGHT", "0.05")),
    resume=env("H1R_EDGE_RESUME", "1") == "1",
    amp=env("H1R_EDGE_AMP", "1") == "1",
)
result = run(args)
summary = {
    "best": result["best"],
    "epochs_requested": args.epochs,
    "max_steps": args.max_steps,
    "appearance_dim": args.appearance_dim,
    "triplet_weight": args.triplet_weight,
    "nodes": str(edge_assets.nodes),
    "identity": str(edge_assets.identity),
}
(out / "summary.json").write_text(json.dumps(summary, indent=2))

for name in (
    "edge_predictor_best.pth", "appearance_projector_best.pth",
    "edge_resume.pth", "config.json", "metrics.json", "summary.json",
):
    source = out / name
    if source.is_file():
        shutil.copy2(source, Path("/kaggle/working") / name)

print("H1R_EDGE_RESULT", json.dumps(summary, sort_keys=True))
